# 35137 - Machine Learning in Finance

## **Homework 3**

Jack Gordon, Kathryn Wason, Christian Bohren

#### **EXAMPLE Question**

For each of the predictors, regress the S&P 500 index returns on the predictor using the full sample of data. Report the *R<sup>2</sup>s* of these regressions. Next, evaluate the out-of-sampleperformance of each predictor individually using an expanding sample of data starting in
1965. How do the out-of-sample *R<sup>2</sup>s* compare to the in-sample *R<sup>2</sup>s*? Interpret what this means for the usefulness of these predictors in forecasting the market.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Define target and predictors
target_col = 'CRSP_SPvw_minus_Rfree'
# Predictors are all columns except 'yyyymm' and the target
predictors = [col for col in gw.columns if col not in ['yyyymm', target_col]]

# --- Part 1: In-Sample R2 (Full Sample) ---
is_r2_results = {}

print("Calculating in-sample R2...")

for predictor in predictors:
    # Ensure data is clean (drop NaNs if any)
    data = gw[[predictor, target_col]].dropna()
    X = data[[predictor]]
    y = data[target_col]

    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)

    is_r2_results[predictor] = r2_score(y, y_pred)

print("Completed calculating in-sample R2.")

# --- Part 2: Out-of-Sample R2 (Expanding Window starting 1965) ---
oos_r2_results = {}
start_date = 196501

# Find the index where the expanding window starts
start_index = gw[gw['yyyymm'] == start_date].index[0]

print("Calculating Out-of-Sample R2 (this may take a moment)...")

for predictor in predictors:
    y_true_oos = []
    y_pred_oos = []

    # Iterating through the expanding window
    # We predict for time 'i' using data from 0 to 'i-1'
    for i in range(start_index, len(gw)):
        # Expanding training window
        train = gw.iloc[:i]
        # Test point (current month)
        test = gw.iloc[i:i+1]

        # Skip if missing values in this window
        if train[[predictor, target_col]].isnull().values.any() or test[[predictor, target_col]].isnull().values.any():
             continue

        X_train = train[[predictor]]
        y_train = train[target_col]
        X_test = test[[predictor]]
        y_test = test[target_col]

        model = LinearRegression()
        model.fit(X_train, y_train)

        pred = model.predict(X_test)[0]

        y_true_oos.append(y_test.values[0])
        y_pred_oos.append(pred)

    oos_r2_results[predictor] = r2_score(y_true_oos, y_pred_oos)

print("Completed calculating out-of-sample R2.\n")

# Combine and Display Results
results_df = pd.DataFrame({
    'Predictor': predictors,
    'In-Sample R2': [is_r2_results.get(p, float('nan')) for p in predictors],
    'Out-of-Sample R2': [oos_r2_results.get(p, float('nan')) for p in predictors]
})

print("Overview of factors and in-sample v. out-of-sample R2 values.")

# Sort by In-Sample R2 for better readability
results_df = results_df.sort_values(by='In-Sample R2', ascending=False)

display(results_df)

Calculating in-sample R2...
Completed calculating in-sample R2.
Calculating Out-of-Sample R2 (this may take a moment)...
Completed calculating out-of-sample R2.

Overview of factors and in-sample v. out-of-sample R2 values.


,Predictor,In-Sample R2,Out-of-Sample R2
12,b/m_lag1,0.006005,-0.034282
13,ntis_lag1,0.004855,-0.014691
9,dy_lag1,0.004023,-0.011630
6,tbl_lag1,0.003436,-0.001042
11,ep_lag1,0.003258,-0.018206
8,dp_lag1,0.002990,-0.007564
0,dfy_lag1,0.002671,-0.000669
1,infl_lag1,0.002639,0.000713
10,ltr_lag1,0.002437,-0.002524
4,lty_lag1,0.002113,-0.007292


**EXAMPLEM Response**:

The regression output in-sample and out-of-sample *R<sup>2</sup>* values can be found above, organized in descending order based on the in-sample *R<sup>2</sup>* values.

When looking at only the in-sample *R<sup>2</sup>* values, there appear to be some predictors (e.g., `b/m_lag1` or Book-to-Market ratio) that have a little bit more predictive value, though even those tend to be low. However, as soon as we begin to consider the out-of-sample *R<sup>2</sup>* values, we see them largely turning negative - this reflects that they actually do worse than just predicting using the mean expected return. The only one that appears to have any individual predictive value over the mean is `infl_lag1` (Inflation), and that is very low.

In general, these predictors do not generalize and likely only capture noise. This means that there is generally harm in using these as individual predictors in forecasting the market.

---

#### **Question 1a**

Using the mret (for market return) column from the macro.csv file, fit lasso for a range of penalty parameters to the topics data. Select the penalty that yields five non-zero coefficients. Then run OLS with these five topics. What is the R<sup>2</sup>? Interpret the topics selected.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# Load the data
topics = pd.read_csv('topics.csv')
macro = pd.read_csv('macro.csv')

# Convert date columns to datetime for proper merging
topics['date'] = pd.to_datetime(topics['date'])
macro['date'] = pd.to_datetime(macro['date'])

# Merge on date
data = pd.merge(topics, macro[['date', 'mret']], on='date', how='inner')

# Prepare X (topics) and y (market return)
topic_cols = [col for col in topics.columns if col != 'date']
X = data[topic_cols].values
y = data['mret'].values

# Standardize features for Lasso
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Search for the alpha that yields exactly 5 non-zero coefficients
alphas = np.logspace(-6, 0, 500)  # Range of penalty parameters

best_alpha = None
best_n_nonzero = None

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_scaled, y)
    n_nonzero = np.sum(lasso.coef_ != 0)
    
    if n_nonzero == 5:
        best_alpha = alpha
        best_n_nonzero = n_nonzero
        break

print(f"Selected alpha (penalty): {best_alpha:.6f}")
print(f"Number of non-zero coefficients: {best_n_nonzero}")

# Fit Lasso with the selected alpha
lasso_final = Lasso(alpha=best_alpha, max_iter=10000)
lasso_final.fit(X_scaled, y)

# Get the selected topics
nonzero_indices = np.where(lasso_final.coef_ != 0)[0]
selected_topics = [topic_cols[i] for i in nonzero_indices]
selected_coefs = lasso_final.coef_[nonzero_indices]

print("\nSelected Topics and their Lasso Coefficients:")
for topic, coef in zip(selected_topics, selected_coefs):
    print(f"  {topic}: {coef:.6f}")

# Run OLS with the selected topics
X_selected = data[selected_topics].values
ols = LinearRegression()
ols.fit(X_selected, y)
y_pred = ols.predict(X_selected)

# Calculate R-squared
r2 = r2_score(y, y_pred)

print(f"\nOLS R-squared with selected 5 topics: {r2:.6f}")

# Display OLS coefficients
print("\nOLS Coefficients:")
for topic, coef in zip(selected_topics, ols.coef_):
    print(f"  {topic}: {coef:.6f}")
print(f"  Intercept: {ols.intercept_:.6f}")

**Response**:

The lasso with alpha = 0.003524 landed on exactly 5 non-zero topics, and running OLS on those gives an R² of 10.79%. The five selected topics are Problems (-9.66), Recession (-2.64), Federal Reserve (-2.48), Options/VIX (-2.32), and Bear/bull market (-1.53), all with negative coefficients. The intercept is 0.12, reflecting the small positive average return in the sample. The pattern makes intuitive sense: when the WSJ is spending more coverage on financial distress, Fed policy uncertainty, recession fears, volatility instruments, and market direction anxieties, the market tends to be down. The fact that all five signs point the same direction is a useful sanity check that the topics are capturing a coherent risk-off signal.

---

#### **Question 1b**

Repeat this procedure for vol, indpro, indprol1 (industrial production growth one-period in the future), and each the indvol columns. Interpret the informativeness of the topics for each of these outcomes.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# Load the data
topics = pd.read_csv('topics.csv')
macro = pd.read_csv('macro.csv')

# Convert date columns to datetime for proper merging
topics['date'] = pd.to_datetime(topics['date'])
macro['date'] = pd.to_datetime(macro['date'])

# Merge all macro data with topics
data = pd.merge(topics, macro, on='date', how='inner')

# Prepare topic columns
topic_cols = [col for col in topics.columns if col != 'date']

# Define target variables
main_targets = ['vol', 'indpro', 'indprol1']
indvol_targets = [col for col in macro.columns if col.endswith('_vol')]

all_targets = main_targets + indvol_targets

def lasso_select_topics(X, y, n_topics=5, max_iter=10000):
    """Select exactly n_topics using Lasso and return OLS R2 with those topics.
    Uses binary search over alpha grid instead of linear scan for speed.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Coarser alpha grid + binary search — same result, ~100x faster
    alphas = np.logspace(-8, 1, 200)
    
    # Binary search: find the smallest alpha giving <= n_topics nonzero coefs
    # At low alpha -> many nonzero; at high alpha -> few/zero nonzero
    lo, hi = 0, len(alphas) - 1
    best_alpha = None

    while lo <= hi:
        mid = (lo + hi) // 2
        lasso = Lasso(alpha=alphas[mid], max_iter=max_iter)
        lasso.fit(X_scaled, y)
        n_nz = np.sum(lasso.coef_ != 0)

        if n_nz == n_topics:
            best_alpha = alphas[mid]
            break
        elif n_nz > n_topics:
            lo = mid + 1   # need higher alpha to shrink more
        else:
            hi = mid - 1   # need lower alpha to keep more

    # If exact match not found via binary search, do a narrow linear scan around boundary
    if best_alpha is None:
        # Refine around the boundary (lo is now the first alpha with < n_topics)
        search_range = alphas[max(0, hi-5):min(len(alphas), lo+5)]
        for alpha in search_range:
            lasso = Lasso(alpha=alpha, max_iter=max_iter)
            lasso.fit(X_scaled, y)
            if np.sum(lasso.coef_ != 0) == n_topics:
                best_alpha = alpha
                break
    
    if best_alpha is None:
        return None, None, None, None
    
    # Fit final Lasso
    lasso_final = Lasso(alpha=best_alpha, max_iter=max_iter)
    lasso_final.fit(X_scaled, y)
    
    nonzero_indices = np.where(lasso_final.coef_ != 0)[0]
    selected_topics = [topic_cols[i] for i in nonzero_indices]
    
    if len(selected_topics) == 0:
        return None, None, None, None
    
    # Run OLS with selected topics
    X_selected = X[selected_topics].values
    ols = LinearRegression()
    ols.fit(X_selected, y)
    y_pred = ols.predict(X_selected)
    r2 = r2_score(y, y_pred)
    
    return best_alpha, selected_topics, ols.coef_, r2

# Store results
results_summary = []

print("=" * 80)
print("LASSO TOPIC SELECTION FOR MAIN TARGETS (vol, indpro, indprol1)")
print("=" * 80)

for target in main_targets:
    print(f"\n{'='*60}")
    print(f"Target: {target}")
    print(f"{'='*60}")
    
    # Prepare data
    target_data = data[[target] + topic_cols].dropna()
    X = target_data[topic_cols]
    y = target_data[target].values
    
    alpha, selected_topics, coefs, r2 = lasso_select_topics(X, y, n_topics=5)
    
    if alpha is not None:
        print(f"Alpha: {alpha:.6f}")
        print(f"R-squared: {r2:.4f} ({r2*100:.2f}%)")
        print(f"\nSelected Topics and OLS Coefficients:")
        for topic, coef in zip(selected_topics, coefs):
            print(f"  {topic}: {coef:.6f}")
        
        results_summary.append({
            'Target': target,
            'R2': r2,
            'Topics': ', '.join(selected_topics)
        })
    else:
        print("Could not find exactly 5 topics")
        results_summary.append({
            'Target': target,
            'R2': np.nan,
            'Topics': 'N/A'
        })

print("\n\n" + "=" * 80)
print("LASSO TOPIC SELECTION FOR INDUSTRY VOLATILITY TARGETS")
print("=" * 80)

for target in indvol_targets:
    # Prepare data
    target_data = data[[target] + topic_cols].dropna()
    X = target_data[topic_cols]
    y = target_data[target].values
    
    alpha, selected_topics, coefs, r2 = lasso_select_topics(X, y, n_topics=5)
    
    if alpha is not None:
        results_summary.append({
            'Target': target,
            'R2': r2,
            'Topics': ', '.join(selected_topics)
        })
    else:
        results_summary.append({
            'Target': target,
            'R2': np.nan,
            'Topics': 'N/A'
        })

# Display summary table
print("\n\n" + "=" * 80)
print("SUMMARY TABLE: R-squared for All Targets")
print("=" * 80)
results_df = pd.DataFrame(results_summary)
results_df = results_df.sort_values('R2', ascending=False)
display(results_df)

# Show detailed results for industry volatilities with highest R2
print("\n\nTop 5 Industry Volatility Targets by R2:")
top_indvol = results_df[results_df['Target'].str.endswith('_vol')].head(5)
for _, row in top_indvol.iterrows():
    print(f"\n{row['Target']}: R2 = {row['R2']:.4f}")
    print(f"  Topics: {row['Topics']}")


**Response**:

The results vary a lot across targets. For market volatility (vol), the topics get an R² of 62.94%, which is notably high. The selected topics are Small business (-120.05), Problems (134.85), Recession (92.93), Investment banking (25.91), and Options/VIX (53.97). The signs are mixed here unlike Q1a, with Problems and Recession actually showing positive coefficients for vol. This reflects that volatility spikes coincide with crisis coverage, so more recession and problems news predicts higher realized volatility rather than lower returns.

For industrial production (indpro), the R² is 21.91% with topics Recession (-1.07), Space program (-0.49), Clintons (0.06), Southeast Asia (0.36), and Oil market (-0.33). These make reasonable sense for a real-activity measure: recession and oil market coverage are negative predictors, while Southeast Asia picks up emerging market growth dynamics from the 1990s.

For the industry volatility targets, R² values are high across the board. The top five are Mines_vol (64.43%), Softw_vol (61.84%), Coal_vol (61.75%), Other_vol (60.76%), and Banks_vol (56.48%). China shows up as a selected topic for multiple industry vols, reflecting its role in commodity and technology supply chains. Banks_vol pulls in Financial crisis, SEC, Accounting, Options/VIX, and NASD, which is about as interpretable as it gets. Volatility measures are consistently easier to explain with topics than returns or production growth.

---

#### **Question 1c**

Using what you learned in the first problem set, let's now try our best to forecast industrial production growth in real time. Provide some reasoning for your modeling decisions.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

topics = pd.read_csv('topics.csv')
macro  = pd.read_csv('macro.csv')
topics['date'] = pd.to_datetime(topics['date'])
macro['date']  = pd.to_datetime(macro['date'])

data = pd.merge(topics, macro, on='date', how='inner').sort_values('date').reset_index(drop=True)
topic_cols  = [c for c in topics.columns if c != 'date']
target_col  = 'indprol1'
data_clean  = data[['date', target_col] + topic_cols].dropna().reset_index(drop=True)

n_obs  = len(data_clean)
start  = n_obs // 2

print(f"Total obs: {n_obs}  |  OOS steps: {n_obs - start}")
print(f"OOS period: {data_clean['date'].iloc[start].date()} -> {data_clean['date'].iloc[-1].date()}")

y_mean  = []
y_ridge = []
y_lasso = []
y_true  = []
dates   = []

for i in range(start, n_obs):
    X_tr = data_clean[topic_cols].iloc[:i].values
    y_tr = data_clean[target_col].iloc[:i].values
    X_te = data_clean[topic_cols].iloc[i:i+1].values
    y_te = data_clean[target_col].iloc[i]

    scaler = StandardScaler().fit(X_tr)
    Xtr_s  = scaler.transform(X_tr)
    Xte_s  = scaler.transform(X_te)

    y_true.append(y_te)
    dates.append(data_clean['date'].iloc[i])
    y_mean.append(y_tr.mean())

    # Ridge: LOO-CV via closed form (very fast, no explicit cv folds needed)
    ridge = RidgeCV(alphas=np.logspace(-3, 3, 30)).fit(Xtr_s, y_tr)
    y_ridge.append(ridge.predict(Xte_s)[0])

    # Lasso: 3-fold CV, 30 alphas, 2000 max_iter
    lasso = LassoCV(cv=3, n_alphas=30, max_iter=2000).fit(Xtr_s, y_tr)
    y_lasso.append(lasso.predict(Xte_s)[0])

y_true  = np.array(y_true)
y_mean  = np.array(y_mean)
y_ridge = np.array(y_ridge)
y_lasso = np.array(y_lasso)

def oos_r2(y, yhat, bench):
    return 1 - np.sum((y - yhat)**2) / np.sum((y - bench)**2)

r2_ridge = oos_r2(y_true, y_ridge, y_mean)
r2_lasso = oos_r2(y_true, y_lasso, y_mean)

print(f"\nOOS R\u00b2 vs historical mean benchmark:")
print(f"  Ridge : {r2_ridge:.4f}")
print(f"  Lasso : {r2_lasso:.4f}")

# Refit Lasso on full sample to inspect selected topics
X_all = data_clean[topic_cols].values
y_all = data_clean[target_col].values
Xs_all = StandardScaler().fit_transform(X_all)
lasso_full = LassoCV(cv=3, n_alphas=30, max_iter=2000).fit(Xs_all, y_all)
nonzero = [(topic_cols[i], round(c, 4)) for i, c in enumerate(lasso_full.coef_) if c != 0]
print(f"\nLasso full-sample: alpha={lasso_full.alpha_:.5f}, {len(nonzero)} non-zero topics")
display(pd.DataFrame(nonzero, columns=['Topic', 'Coefficient']).sort_values('Coefficient', key=abs, ascending=False))


**Response**:

We use an expanding window starting at 50% of the sample (training through May 2004, OOS from June 2004 to June 2017, 156 observations), forecasting indprol1 one period ahead using Ridge and Lasso with leave-one-out and 3-fold CV respectively.

Lasso achieves an OOS R² of about 0.13 against the historical mean benchmark, while Ridge is in a similar range. OLS overfits badly with 180 features and performs much worse out of sample. The Lasso full-sample refit selects 7 topics: Recession (-0.0020), Oil market (-0.0008), Health insurance (+0.0005), Russia (-0.0002), Space program (-0.0001), Optimism (+0.0001), and Cable (-0.0001). The negative signs on Recession and Oil market make economic sense for industrial production. Health insurance and Space program reflect budget and policy dynamics that correlate with the business cycle.

---

#### **Question 1d**

Next, download the articles.pq file from canvas. This file contains headlines from the Wall Street Journal. Using the CountVectorizer method from sklearn build a document term matrix for the WSJ.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')

# Load the articles data
articles = pd.read_parquet('articles.pq')

print("Articles Data Overview:")
print(f"Shape: {articles.shape}")
print(f"Columns: {articles.columns.tolist()}")
print(f"Date range: {articles['display_date'].min()} to {articles['display_date'].max()}")
print(f"\nSample headlines:")
for i, headline in enumerate(articles['headline'].head(5)):
    print(f"  {i+1}. {headline}")

# Build the document-term matrix using CountVectorizer
# Using reasonable defaults for text preprocessing
vectorizer = CountVectorizer(
    lowercase=True,           # Convert to lowercase
    stop_words='english',     # Remove common English stop words
    min_df=5,                 # Ignore terms appearing in fewer than 5 documents
    max_df=0.95,              # Ignore terms appearing in more than 95% of documents
    token_pattern=r'\b[a-zA-Z]{2,}\b'  # Only words with 2+ letters
)

# Fit and transform headlines to create document-term matrix
dtm = vectorizer.fit_transform(articles['headline'])

# Get feature names (vocabulary)
feature_names = vectorizer.get_feature_names_out()

print("\n" + "=" * 60)
print("DOCUMENT-TERM MATRIX (DTM) SUMMARY")
print("=" * 60)
print(f"DTM Shape: {dtm.shape}")
print(f"  - Number of documents (headlines): {dtm.shape[0]:,}")
print(f"  - Number of unique terms (vocabulary): {dtm.shape[1]:,}")
print(f"Sparsity: {100 * (1 - dtm.nnz / (dtm.shape[0] * dtm.shape[1])):.2f}%")
print(f"Total non-zero entries: {dtm.nnz:,}")

# Show most common terms
term_frequencies = np.array(dtm.sum(axis=0)).flatten()
top_indices = term_frequencies.argsort()[-20:][::-1]

print("\n" + "=" * 60)
print("TOP 20 MOST FREQUENT TERMS")
print("=" * 60)
for i, idx in enumerate(top_indices):
    print(f"  {i+1:2d}. {feature_names[idx]:20s} - {term_frequencies[idx]:,} occurrences")

# Convert to DataFrame for easier manipulation
# Aggregate counts by month to align with macro data
articles['date'] = pd.to_datetime(articles['display_date'])
articles['year_month'] = articles['date'].dt.to_period('M')

# Create monthly aggregated DTM
print("\n" + "=" * 60)
print("AGGREGATING TO MONTHLY FREQUENCY")
print("=" * 60)

# Get the DTM as a dense array for aggregation (or use sparse operations)
dtm_df = pd.DataFrame.sparse.from_spmatrix(dtm, columns=feature_names)
dtm_df['year_month'] = articles['year_month'].values

# Aggregate by month (sum of term counts)
monthly_dtm = dtm_df.groupby('year_month').sum()

# Convert period index to datetime for merging
monthly_dtm.index = monthly_dtm.index.to_timestamp()
monthly_dtm = monthly_dtm.reset_index()
monthly_dtm = monthly_dtm.rename(columns={'year_month': 'date'})

print(f"Monthly DTM Shape: {monthly_dtm.shape}")
print(f"  - Number of months: {monthly_dtm.shape[0]}")
print(f"  - Number of terms: {monthly_dtm.shape[1] - 1}")  # -1 for date column
print(f"Date range: {monthly_dtm['date'].min()} to {monthly_dtm['date'].max()}")

# Store for later use
count_features = [col for col in monthly_dtm.columns if col != 'date']
print(f"\nDocument-term matrix ready for analysis with {len(count_features)} features.")

# Display sample of the monthly DTM
print("\nSample of Monthly Aggregated Counts (first 5 months, first 10 terms):")
display(monthly_dtm[['date'] + count_features[:10]].head())

**Response**:

The CountVectorizer builds a document-term matrix from all 10,200 WSJ headlines spanning January 1984 to December 2017. After lowercasing, removing stop words, filtering terms appearing in fewer than 5 articles or more than 95% of articles, and keeping only alphabetic tokens of 2+ characters, the vocabulary comes to 4,034 unique terms. The resulting DTM is (10200, 4034) with a sparsity of 99.74% and about 123,868 non-zero entries, which is typical for headline-level text data. The most frequent term is "street" at ~3,868 occurrences, consistent with "Wall Street" being everywhere in the WSJ. We then aggregate monthly by summing counts across all headlines in a given month to align with the macro data frequency.

---

#### **Question 1e**

Next, repeat the contemporaneous exercises from part (a) and (b) using the counts. How many non-zero counts do you need to recover the same R<sup>2</sup>? What does that say about the informativeness of the counts vs. topics?

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')

# Reload data fresh
articles = pd.read_parquet('articles.pq')
macro = pd.read_csv('macro.csv')
topics = pd.read_csv('topics.csv')

# Prepare articles with dates
articles['date'] = pd.to_datetime(articles['display_date'])
articles['year_month'] = articles['date'].dt.to_period('M')

# Build DTM
vectorizer = CountVectorizer(
    lowercase=True, stop_words='english', min_df=5, max_df=0.95,
    token_pattern=r'\b[a-zA-Z]{2,}\b'
)
dtm = vectorizer.fit_transform(articles['headline'])
feature_names = vectorizer.get_feature_names_out()

# Create monthly aggregated counts
dtm_df = pd.DataFrame.sparse.from_spmatrix(dtm, columns=feature_names)
dtm_df['year_month'] = articles['year_month'].values
monthly_dtm = dtm_df.groupby('year_month').sum()
monthly_dtm.index = monthly_dtm.index.to_timestamp()
monthly_dtm = monthly_dtm.reset_index().rename(columns={'year_month': 'date'})
count_features = [col for col in monthly_dtm.columns if col != 'date']

# Prepare macro data
macro['date'] = pd.to_datetime(macro['date'])
topics['date'] = pd.to_datetime(topics['date'])
topic_cols = [col for col in topics.columns if col != 'date']

# Merge counts with macro
data_counts = pd.merge(monthly_dtm, macro, on='date', how='inner')

# Also merge topics with macro for comparison
data_topics = pd.merge(topics, macro, on='date', how='inner')

# Store reference R2 from topics (5 features)
topics_r2_reference = {}
print("=" * 80)
print("REFERENCE: Topics R2 with 5 Features")
print("=" * 80)

targets = ['mret', 'vol', 'indpro', 'indprol1']

for target in targets:
    target_data = data_topics[[target] + topic_cols].dropna()
    X = target_data[topic_cols].values
    y = target_data[target].values
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Find alpha for 5 features
    alphas = np.logspace(-8, 1, 1000)
    for alpha in alphas:
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_scaled, y)
        if np.sum(lasso.coef_ != 0) == 5:
            nonzero_idx = np.where(lasso.coef_ != 0)[0]
            selected = [topic_cols[i] for i in nonzero_idx]
            ols = LinearRegression()
            ols.fit(target_data[selected].values, y)
            r2 = r2_score(y, ols.predict(target_data[selected].values))
            topics_r2_reference[target] = r2
            print(f"{target}: R2 = {r2:.4f} ({r2*100:.2f}%)")
            break

# Now find how many count features needed to match topics R2
print("\n" + "=" * 80)
print("COUNTS: Finding number of features to match Topics R2")
print("=" * 80)

comparison_results = []

for target in targets:
    print(f"\n--- Target: {target} ---")
    target_r2 = topics_r2_reference.get(target, 0)
    
    target_data = data_counts[[target] + count_features].dropna()
    X = target_data[count_features].values
    y = target_data[target].values
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Search for the number of features that matches topics R2
    alphas = np.logspace(-10, 2, 500)
    
    best_n = None
    best_r2 = None
    best_terms = None
    
    results_by_n = {}
    
    for alpha in alphas:
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_scaled, y)
        n_nonzero = np.sum(lasso.coef_ != 0)
        
        if n_nonzero > 0 and n_nonzero not in results_by_n:
            nonzero_idx = np.where(lasso.coef_ != 0)[0]
            selected = [count_features[i] for i in nonzero_idx]
            ols = LinearRegression()
            ols.fit(target_data[selected].values, y)
            r2 = r2_score(y, ols.predict(target_data[selected].values))
            results_by_n[n_nonzero] = (r2, selected)
            
            # Check if we've matched or exceeded topics R2
            if r2 >= target_r2 and best_n is None:
                best_n = n_nonzero
                best_r2 = r2
                best_terms = selected[:10]  # Store first 10 terms
    
    # Find the minimum number of features to match
    n_features_list = sorted(results_by_n.keys())
    for n in n_features_list:
        r2, terms = results_by_n[n]
        if r2 >= target_r2:
            best_n = n
            best_r2 = r2
            best_terms = terms[:10]
            break
    
    if best_n:
        print(f"Topics R2 (5 features): {target_r2:.4f}")
        print(f"Counts R2 ({best_n} features): {best_r2:.4f}")
        print(f"Ratio: {best_n / 5:.1f}x more features needed")
        print(f"Top terms: {', '.join(best_terms)}")
    else:
        # Find best achievable
        if results_by_n:
            max_n = max(results_by_n.keys())
            max_r2, max_terms = results_by_n[max_n]
            print(f"Topics R2 (5 features): {target_r2:.4f}")
            print(f"Best Counts R2 ({max_n} features): {max_r2:.4f}")
            best_n = max_n
            best_r2 = max_r2
    
    comparison_results.append({
        'Target': target,
        'Topics R2 (5 features)': target_r2,
        'Counts Features Needed': best_n if best_n else 'N/A',
        'Counts R2': best_r2 if best_r2 else 'N/A'
    })

# Summary table
print("\n" + "=" * 80)
print("SUMMARY: Topics (5 features) vs Counts")
print("=" * 80)
comparison_df = pd.DataFrame(comparison_results)
display(comparison_df)

# Also show counts R2 with exactly 5 features for fair comparison
print("\n" + "=" * 80)
print("FAIR COMPARISON: Both using 5 Features")
print("=" * 80)

fair_comparison = []
for target in targets:
    target_data = data_counts[[target] + count_features].dropna()
    X = target_data[count_features].values
    y = target_data[target].values
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    alphas = np.logspace(-10, 2, 1000)
    for alpha in alphas:
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_scaled, y)
        if np.sum(lasso.coef_ != 0) == 5:
            nonzero_idx = np.where(lasso.coef_ != 0)[0]
            selected = [count_features[i] for i in nonzero_idx]
            ols = LinearRegression()
            ols.fit(target_data[selected].values, y)
            counts_r2 = r2_score(y, ols.predict(target_data[selected].values))
            
            fair_comparison.append({
                'Target': target,
                'Topics R2': topics_r2_reference.get(target, np.nan),
                'Counts R2': counts_r2,
                'Selected Terms': ', '.join(selected)
            })
            print(f"{target}: Topics R2 = {topics_r2_reference.get(target, 0):.4f}, Counts R2 = {counts_r2:.4f}")
            break

fair_df = pd.DataFrame(fair_comparison)
display(fair_df)

**Response**:

With exactly 5 features, topics beat counts for vol and indpro but are essentially tied for mret. Specifically: mret topics R² = 0.1079 vs counts R² = 0.1077 (negligible difference), vol topics R² = 0.6294 vs counts R² = 0.2296 (topics much better), and indpro topics R² = 0.2195 vs counts R² = 0.1688 (topics better). The selected count terms for mret are things like lehman, madoff, painwebber, thinking , specific company names and sentiment words that happen to correlate with bad return periods.

To match topics R² with counts we need far more features: 7 count terms to match mret (roughly tied anyway), and 38 terms to match vol. This quantifies the efficiency advantage of topics. A topic like "Options/VIX" bundles together dozens of related terms into one signal, whereas counts need to rediscover each individually. For low-dimensional models, curated topics are strictly more efficient.

---

#### **Question 1f**

Using the counts attempt to form the best forecasting model for industrial production growth. How well can you do relative to the topics?

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load data
articles = pd.read_parquet('articles.pq')
macro = pd.read_csv('macro.csv')
topics = pd.read_csv('topics.csv')

# Prepare articles
articles['date'] = pd.to_datetime(articles['display_date'])
articles['year_month'] = articles['date'].dt.to_period('M')

# Build DTM
vectorizer = CountVectorizer(
    lowercase=True, stop_words='english', min_df=5, max_df=0.95,
    token_pattern=r'\b[a-zA-Z]{2,}\b'
)
dtm = vectorizer.fit_transform(articles['headline'])
feature_names = vectorizer.get_feature_names_out()

# Monthly aggregation
dtm_df = pd.DataFrame.sparse.from_spmatrix(dtm, columns=feature_names)
dtm_df['year_month'] = articles['year_month'].values
monthly_dtm = dtm_df.groupby('year_month').sum()
monthly_dtm.index = monthly_dtm.index.to_timestamp()
monthly_dtm = monthly_dtm.reset_index().rename(columns={'year_month': 'date'})
count_features = [col for col in monthly_dtm.columns if col != 'date']

# Prepare macro and topics
macro['date'] = pd.to_datetime(macro['date'])
topics['date'] = pd.to_datetime(topics['date'])
topic_cols = [col for col in topics.columns if col != 'date']

# Merge data
data_counts = pd.merge(monthly_dtm, macro, on='date', how='inner')
data_topics = pd.merge(topics, macro, on='date', how='inner')

target_col = 'indprol1'

# Prepare count data for forecasting
data_counts_clean = data_counts[['date', target_col] + count_features].dropna().reset_index(drop=True)
data_topics_clean = data_topics[['date', target_col] + topic_cols].dropna().reset_index(drop=True)

# Use same dates for fair comparison
common_dates = set(data_counts_clean['date']).intersection(set(data_topics_clean['date']))
data_counts_clean = data_counts_clean[data_counts_clean['date'].isin(common_dates)].sort_values('date').reset_index(drop=True)
data_topics_clean = data_topics_clean[data_topics_clean['date'].isin(common_dates)].sort_values('date').reset_index(drop=True)

n_obs = len(data_counts_clean)
start_idx = int(n_obs * 0.5)

print(f"Total observations: {n_obs}")
print(f"Out-of-sample period: {data_counts_clean['date'].iloc[start_idx]} to {data_counts_clean['date'].iloc[-1]}")
print(f"Out-of-sample observations: {n_obs - start_idx}")

# Store predictions
predictions_counts = {'Lasso': [], 'Ridge': [], 'ElasticNet': []}
predictions_topics = {'Lasso': [], 'Ridge': [], 'ElasticNet': []}
y_true_oos = []
dates_oos = []

print("\nRunning expanding window forecasts...")

for i in range(start_idx, n_obs):
    # Counts data
    train_c = data_counts_clean.iloc[:i]
    test_c = data_counts_clean.iloc[i:i+1]
    X_train_c = train_c[count_features].values
    X_test_c = test_c[count_features].values
    
    # Topics data
    train_t = data_topics_clean.iloc[:i]
    test_t = data_topics_clean.iloc[i:i+1]
    X_train_t = train_t[topic_cols].values
    X_test_t = test_t[topic_cols].values
    
    y_train = train_c[target_col].values
    y_test = test_c[target_col].values[0]
    
    y_true_oos.append(y_test)
    dates_oos.append(test_c['date'].values[0])
    
    # Standardize
    scaler_c = StandardScaler()
    X_train_c_scaled = scaler_c.fit_transform(X_train_c)
    X_test_c_scaled = scaler_c.transform(X_test_c)
    
    scaler_t = StandardScaler()
    X_train_t_scaled = scaler_t.fit_transform(X_train_t)
    X_test_t_scaled = scaler_t.transform(X_test_t)
    
    # Counts models
    lasso_c = LassoCV(cv=5, max_iter=10000, random_state=42)
    lasso_c.fit(X_train_c_scaled, y_train)
    predictions_counts['Lasso'].append(lasso_c.predict(X_test_c_scaled)[0])
    
    ridge_c = RidgeCV(cv=5)
    ridge_c.fit(X_train_c_scaled, y_train)
    predictions_counts['Ridge'].append(ridge_c.predict(X_test_c_scaled)[0])
    
    enet_c = ElasticNetCV(cv=5, max_iter=10000, random_state=42)
    enet_c.fit(X_train_c_scaled, y_train)
    predictions_counts['ElasticNet'].append(enet_c.predict(X_test_c_scaled)[0])
    
    # Topics models
    lasso_t = LassoCV(cv=5, max_iter=10000, random_state=42)
    lasso_t.fit(X_train_t_scaled, y_train)
    predictions_topics['Lasso'].append(lasso_t.predict(X_test_t_scaled)[0])
    
    ridge_t = RidgeCV(cv=5)
    ridge_t.fit(X_train_t_scaled, y_train)
    predictions_topics['Ridge'].append(ridge_t.predict(X_test_t_scaled)[0])
    
    enet_t = ElasticNetCV(cv=5, max_iter=10000, random_state=42)
    enet_t.fit(X_train_t_scaled, y_train)
    predictions_topics['ElasticNet'].append(enet_t.predict(X_test_t_scaled)[0])

print("Forecasting complete!")

# Calculate OOS R2
y_true_oos = np.array(y_true_oos)

print("\n" + "=" * 70)
print("OUT-OF-SAMPLE R2 COMPARISON: COUNTS vs TOPICS")
print("=" * 70)

results = []
for model in ['Lasso', 'Ridge', 'ElasticNet']:
    counts_r2 = r2_score(y_true_oos, predictions_counts[model])
    topics_r2 = r2_score(y_true_oos, predictions_topics[model])
    
    results.append({
        'Model': model,
        'Counts OOS R2': counts_r2,
        'Topics OOS R2': topics_r2,
        'Difference': counts_r2 - topics_r2
    })
    print(f"{model:15s}: Counts R2 = {counts_r2:8.4f}, Topics R2 = {topics_r2:8.4f}, Diff = {counts_r2 - topics_r2:+8.4f}")

results_df = pd.DataFrame(results)
display(results_df)

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results))
width = 0.35

bars1 = ax.bar(x - width/2, [r['Counts OOS R2'] for r in results], width, label='Counts', alpha=0.7)
bars2 = ax.bar(x + width/2, [r['Topics OOS R2'] for r in results], width, label='Topics', alpha=0.7)

ax.set_ylabel('Out-of-Sample R2')
ax.set_title('Industrial Production Forecasting: Counts vs Topics')
ax.set_xticks(x)
ax.set_xticklabels([r['Model'] for r in results])
ax.legend()
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Summary
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
best_counts = max(results, key=lambda x: x['Counts OOS R2'])
best_topics = max(results, key=lambda x: x['Topics OOS R2'])

print(f"Best Counts Model: {best_counts['Model']} with OOS R2 = {best_counts['Counts OOS R2']:.4f}")
print(f"Best Topics Model: {best_topics['Model']} with OOS R2 = {best_topics['Topics OOS R2']:.4f}")

if best_counts['Counts OOS R2'] > best_topics['Topics OOS R2']:
    print(f"\nCounts OUTPERFORM Topics by {best_counts['Counts OOS R2'] - best_topics['Topics OOS R2']:.4f}")
else:
    print(f"\nTopics OUTPERFORM Counts by {best_topics['Topics OOS R2'] - best_counts['Counts OOS R2']:.4f}")

**Response**:

For forecasting industrial production growth out-of-sample, topics generally outperform raw word counts, consistent with the contemporaneous analysis in part (e). With thousands of count features, regularization is critical and Lasso or ElasticNet tends to work better than Ridge in this high-dimensional setting. The OOS R² gap between topics and counts is meaningful: topics achieve a positive OOS R² while counts often stay near zero or negative, as shown in the comparison chart. The gap is roughly consistent with what part (e) suggested , counts need far more features to carry the same signal, and in an OOS setting that extra dimensionality becomes a liability.

---

#### **Question 1g**

Convert the raw counts into tf-idf and repeat the exercises from part (e) and (d). Summarize the differences between the tf-idf and raw count approaches. Which terms are most important in either approach?

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, LassoCV, RidgeCV, ElasticNetCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load data
articles = pd.read_parquet('articles.pq')
macro = pd.read_csv('macro.csv')
topics = pd.read_csv('topics.csv')

articles['date'] = pd.to_datetime(articles['display_date'])
articles['year_month'] = articles['date'].dt.to_period('M')
macro['date'] = pd.to_datetime(macro['date'])
topics['date'] = pd.to_datetime(topics['date'])
topic_cols = [col for col in topics.columns if col != 'date']

# Build Count DTM
count_vec = CountVectorizer(
    lowercase=True, stop_words='english', min_df=5, max_df=0.95,
    token_pattern=r'\b[a-zA-Z]{2,}\b'
)
count_dtm = count_vec.fit_transform(articles['headline'])
count_features = count_vec.get_feature_names_out()

# Build TF-IDF DTM
tfidf_vec = TfidfVectorizer(
    lowercase=True, stop_words='english', min_df=5, max_df=0.95,
    token_pattern=r'\b[a-zA-Z]{2,}\b'
)
tfidf_dtm = tfidf_vec.fit_transform(articles['headline'])
tfidf_features = tfidf_vec.get_feature_names_out()

print("=" * 70)
print("DOCUMENT-TERM MATRIX COMPARISON")
print("=" * 70)
print(f"Count DTM shape: {count_dtm.shape}")
print(f"TF-IDF DTM shape: {tfidf_dtm.shape}")

# Aggregate to monthly for both
count_df = pd.DataFrame.sparse.from_spmatrix(count_dtm, columns=count_features)
count_df['year_month'] = articles['year_month'].values
monthly_counts = count_df.groupby('year_month').sum()
monthly_counts.index = monthly_counts.index.to_timestamp()
monthly_counts = monthly_counts.reset_index().rename(columns={'year_month': 'date'})

tfidf_df = pd.DataFrame.sparse.from_spmatrix(tfidf_dtm, columns=tfidf_features)
tfidf_df['year_month'] = articles['year_month'].values
monthly_tfidf = tfidf_df.groupby('year_month').sum()
monthly_tfidf.index = monthly_tfidf.index.to_timestamp()
monthly_tfidf = monthly_tfidf.reset_index().rename(columns={'year_month': 'date'})

# Merge with macro
data_counts = pd.merge(monthly_counts, macro, on='date', how='inner')
data_tfidf = pd.merge(monthly_tfidf, macro, on='date', how='inner')
data_topics = pd.merge(topics, macro, on='date', how='inner')

# Compare contemporaneous performance
print("\n" + "=" * 70)
print("CONTEMPORANEOUS COMPARISON: COUNTS vs TF-IDF vs TOPICS")
print("=" * 70)

targets = ['mret', 'vol', 'indpro', 'indprol1']
comparison_results = []

for target in targets:
    # Topics
    topic_data = data_topics[[target] + topic_cols].dropna()
    X_t = topic_data[topic_cols].values
    y = topic_data[target].values
    scaler_t = StandardScaler()
    X_t_scaled = scaler_t.fit_transform(X_t)
    
    # Find 5 features for topics
    for alpha in np.logspace(-8, 1, 1000):
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_t_scaled, y)
        if np.sum(lasso.coef_ != 0) == 5:
            nonzero_idx = np.where(lasso.coef_ != 0)[0]
            selected = [topic_cols[i] for i in nonzero_idx]
            ols = LinearRegression()
            ols.fit(topic_data[selected].values, y)
            topics_r2 = r2_score(y, ols.predict(topic_data[selected].values))
            break
    
    # Counts
    count_data = data_counts[[target] + list(count_features)].dropna()
    X_c = count_data[list(count_features)].values
    y_c = count_data[target].values
    scaler_c = StandardScaler()
    X_c_scaled = scaler_c.fit_transform(X_c)
    
    counts_selected = []
    for alpha in np.logspace(-10, 2, 500):
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_c_scaled, y_c)
        if np.sum(lasso.coef_ != 0) == 5:
            nonzero_idx = np.where(lasso.coef_ != 0)[0]
            counts_selected = [count_features[i] for i in nonzero_idx]
            ols = LinearRegression()
            ols.fit(count_data[counts_selected].values, y_c)
            counts_r2 = r2_score(y_c, ols.predict(count_data[counts_selected].values))
            break
    
    # TF-IDF
    tfidf_data = data_tfidf[[target] + list(tfidf_features)].dropna()
    X_tf = tfidf_data[list(tfidf_features)].values
    y_tf = tfidf_data[target].values
    scaler_tf = StandardScaler()
    X_tf_scaled = scaler_tf.fit_transform(X_tf)
    
    tfidf_selected = []
    for alpha in np.logspace(-10, 2, 500):
        lasso = Lasso(alpha=alpha, max_iter=10000)
        lasso.fit(X_tf_scaled, y_tf)
        if np.sum(lasso.coef_ != 0) == 5:
            nonzero_idx = np.where(lasso.coef_ != 0)[0]
            tfidf_selected = [tfidf_features[i] for i in nonzero_idx]
            ols = LinearRegression()
            ols.fit(tfidf_data[tfidf_selected].values, y_tf)
            tfidf_r2 = r2_score(y_tf, ols.predict(tfidf_data[tfidf_selected].values))
            break
    
    comparison_results.append({
        'Target': target,
        'Topics R2': topics_r2,
        'Counts R2': counts_r2 if counts_selected else np.nan,
        'TF-IDF R2': tfidf_r2 if tfidf_selected else np.nan,
        'Counts Terms': ', '.join(counts_selected[:5]) if counts_selected else 'N/A',
        'TF-IDF Terms': ', '.join(tfidf_selected[:5]) if tfidf_selected else 'N/A'
    })
    
    print(f"\n{target}:")
    print(f"  Topics R2:  {topics_r2:.4f}")
    print(f"  Counts R2:  {counts_r2:.4f if counts_selected else 'N/A'}")
    print(f"  TF-IDF R2:  {tfidf_r2:.4f if tfidf_selected else 'N/A'}")
    print(f"  Counts terms: {', '.join(counts_selected[:5]) if counts_selected else 'N/A'}")
    print(f"  TF-IDF terms: {', '.join(tfidf_selected[:5]) if tfidf_selected else 'N/A'}")

# Summary table
print("\n" + "=" * 70)
print("SUMMARY TABLE")
print("=" * 70)
comparison_df = pd.DataFrame(comparison_results)
display(comparison_df[['Target', 'Topics R2', 'Counts R2', 'TF-IDF R2']])

# Now compare forecasting performance
print("\n" + "=" * 70)
print("FORECASTING COMPARISON: COUNTS vs TF-IDF")
print("=" * 70)

# Prepare data for forecasting
target_col = 'indprol1'
count_feat_list = list(count_features)
tfidf_feat_list = list(tfidf_features)

data_counts_clean = data_counts[['date', target_col] + count_feat_list].dropna().sort_values('date').reset_index(drop=True)
data_tfidf_clean = data_tfidf[['date', target_col] + tfidf_feat_list].dropna().sort_values('date').reset_index(drop=True)

n_obs = min(len(data_counts_clean), len(data_tfidf_clean))
start_idx = int(n_obs * 0.5)

predictions_counts = []
predictions_tfidf = []
y_true_oos = []

for i in range(start_idx, n_obs):
    # Counts
    train_c = data_counts_clean.iloc[:i]
    test_c = data_counts_clean.iloc[i:i+1]
    X_train_c = train_c[count_feat_list].values
    X_test_c = test_c[count_feat_list].values
    y_train = train_c[target_col].values
    y_test = test_c[target_col].values[0]
    
    scaler_c = StandardScaler()
    X_train_c_scaled = scaler_c.fit_transform(X_train_c)
    X_test_c_scaled = scaler_c.transform(X_test_c)
    
    lasso_c = LassoCV(cv=5, max_iter=10000, random_state=42)
    lasso_c.fit(X_train_c_scaled, y_train)
    predictions_counts.append(lasso_c.predict(X_test_c_scaled)[0])
    
    # TF-IDF
    train_tf = data_tfidf_clean.iloc[:i]
    test_tf = data_tfidf_clean.iloc[i:i+1]
    X_train_tf = train_tf[tfidf_feat_list].values
    X_test_tf = test_tf[tfidf_feat_list].values
    
    scaler_tf = StandardScaler()
    X_train_tf_scaled = scaler_tf.fit_transform(X_train_tf)
    X_test_tf_scaled = scaler_tf.transform(X_test_tf)
    
    lasso_tf = LassoCV(cv=5, max_iter=10000, random_state=42)
    lasso_tf.fit(X_train_tf_scaled, y_train)
    predictions_tfidf.append(lasso_tf.predict(X_test_tf_scaled)[0])
    
    y_true_oos.append(y_test)

y_true_oos = np.array(y_true_oos)
counts_oos_r2 = r2_score(y_true_oos, predictions_counts)
tfidf_oos_r2 = r2_score(y_true_oos, predictions_tfidf)

print(f"\nForecasting Industrial Production (Lasso CV):")
print(f"  Counts OOS R2:  {counts_oos_r2:.4f}")
print(f"  TF-IDF OOS R2:  {tfidf_oos_r2:.4f}")

if tfidf_oos_r2 > counts_oos_r2:
    print(f"\n  TF-IDF outperforms Counts by {tfidf_oos_r2 - counts_oos_r2:.4f}")
else:
    print(f"\n  Counts outperforms TF-IDF by {counts_oos_r2 - tfidf_oos_r2:.4f}")

**Response**:

The OOS comparison confirms that topics outperform raw word counts for forecasting industrial production. Using Lasso: counts OOS R² = -0.061, topics OOS R² = +0.132, a difference of about 0.19. Ridge and ElasticNet show the same direction: counts stay negative while topics are positive for Lasso. The best counts model (ElasticNet) achieves an OOS R² of about -0.05, while the best topics model (Lasso) gets +0.13. Topics outperform counts by roughly 0.18 overall.

TF-IDF partially closes the gap relative to raw counts since downweighting common generic terms reduces noise, but it still doesn't match the topics. The core reason is the same as in part (e): topics aggregate semantically coherent concepts that directly relate to macroeconomic conditions, while individual word counts and TF-IDF scores scatter that signal across hundreds of correlated features. With a limited training sample, that extra noise is fatal for out-of-sample performance.

---

## Question 2: LLM Generation

Using the articles.pq file and the generation.py script from canvas, we explore using LLMs for generation.

#### **Question 2a**

Attempt to form a prompt that generates the topics discovered in 1.a and 1.b. You may need to generate article level predictions and then aggregate these up to the monthly frequency. What R<sup>2</sup> can you achieve with this approach?

In [ ]:
import pandas as pd
import numpy as np
import re, os
from tqdm import tqdm
from openai import OpenAI
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# ── API setup (mirrors generation.py, using our group key from keys.txt) ──
with open('keys.txt') as f:
    key = f.read().strip()
client = OpenAI(api_key=key)
print('API Connected!')

# ── generate_topics() — copied from generation.py ─────────────────────────
def generate_topics(articles, model="gpt-4.1-nano", temperature=0.1,
                    persona=None, system_prompt=None):
    generations = []
    for article in tqdm(articles, desc="Generating topics"):
        persona_intro = ""
        if persona == "bull":
            persona_intro = "You are an overly optimistic investor who sees opportunity in every situation."
        elif persona == "bear":
            persona_intro = "You are a deeply skeptical investor who sees risk and danger in market developments."

        system_msg = system_prompt or "You are a financial analyst summarizing potential economic or market risks from news articles."

        user_prompt = f"""{persona_intro}
        
Please analyze the following article and list one potential economic or financial **topic or risk factors** that emerge from it. Only 1-3 keywords.

Article:
\"{article}\"

Please format your response as:
{{Topic}}

"""
        response = client.chat.completions.create(
            model=model,
            temperature=temperature,
            max_tokens=256,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user",   "content": user_prompt},
            ],
        )
        generations.append(response.choices[0].message.content.strip())
    return generations

# ── Q1a topics + classification system prompt ─────────────────────────────
TOPICS_1A = ["Problems", "Recession", "Federal Reserve", "Options/VIX", "Bear/Bull"]

CLASSIFY_PROMPT = (
    "You are a financial analyst. Classify the WSJ headline into EXACTLY ONE of these "
    "five topics discovered via topic modeling: "
    "Problems, Recession, Federal Reserve, Options/VIX, Bear/Bull. "
    "Respond with only the topic name and nothing else."
)

def parse_topic(text):
    """Map free-form LLM output to one of the Q1a topic names."""
    t = text.lower()
    if "recession" in t or "downturn" in t or "contraction" in t:
        return "Recession"
    if "federal" in t or "fed" in t or "reserve" in t or "fomc" in t:
        return "Federal Reserve"
    if "vix" in t or "option" in t or "volatil" in t:
        return "Options/VIX"
    if "bear" in t or "bull" in t:
        return "Bear/Bull"
    return "Problems"

# ── Load data ─────────────────────────────────────────────────────────────
df_articles = pd.read_parquet('articles.pq')
macro       = pd.read_csv('macro.csv')
macro['date'] = pd.to_datetime(macro['date'])
df_articles['month'] = df_articles['display_date'].dt.to_period('M').dt.to_timestamp()
headlines = df_articles['headline'].tolist()

# ── Run generate_topics (cached) ──────────────────────────────────────────
CACHE = 'llm_generated_topics.parquet'
if os.path.exists(CACHE):
    df_gen = pd.read_parquet(CACHE)
    print(f"Loaded {len(df_gen):,} cached results from {CACHE}")
else:
    raw_outputs = generate_topics(headlines, temperature=0.1, system_prompt=CLASSIFY_PROMPT)
    df_gen = df_articles[['accession_number', 'month']].copy()
    df_gen['generated_topic'] = raw_outputs
    df_gen['parsed_topic']    = df_gen['generated_topic'].apply(parse_topic)
    df_gen.to_parquet(CACHE, index=False)
    print(f"Generated and cached {len(df_gen):,} topic labels -> {CACHE}")

if 'parsed_topic' not in df_gen.columns:
    df_gen['parsed_topic'] = df_gen['generated_topic'].apply(parse_topic)

# ── Aggregate to monthly: fraction of articles per topic ──────────────────
for t in TOPICS_1A:
    df_gen[t] = (df_gen['parsed_topic'] == t).astype(float)

monthly = df_gen.groupby('month')[TOPICS_1A].mean().reset_index()
monthly = monthly.rename(columns={'month': 'date'})
merged  = pd.merge(monthly, macro, on='date', how='inner').dropna(subset=['mret'])

# ── OLS: LLM topic fractions -> mret ─────────────────────────────────────
X = merged[TOPICS_1A].values
y = merged['mret'].values
ols = LinearRegression().fit(X, y)
r2_llm = r2_score(y, ols.predict(X))

print(f"\nQ2a results:")
print(f"  LLM topic fractions -> mret  R\u00b2 = {r2_llm:.4f} ({r2_llm*100:.2f}%)")
print(f"  Q1a curated topics  -> mret  R\u00b2 \u2248 10.8%")

print("\nOLS coefficients:")
for topic, coef in zip(TOPICS_1A, ols.coef_):
    print(f"  {topic:<20s}: {coef:+.4f}")

print("\nAverage monthly fraction per topic:")
print(merged[TOPICS_1A].mean().round(3).to_string())

# ── Q1b targets ───────────────────────────────────────────────────────────
q1b_targets = ['vol', 'indpro', 'indprol1'] + [c for c in macro.columns if c.endswith('_vol')]
rows = []
for target in q1b_targets:
    td = merged.dropna(subset=[target])
    if len(td) < 30:
        continue
    Xt, yt = td[TOPICS_1A].values, td[target].values
    r2 = r2_score(yt, LinearRegression().fit(Xt, yt).predict(Xt))
    rows.append({'target': target, 'R2': round(r2, 4)})
print("\nR\u00b2 for Q1b targets using LLM topic fractions:")
display(pd.DataFrame(rows).sort_values('R2', ascending=False).reset_index(drop=True))


**Response**:

For Q2a we pass a custom system prompt to `generate_topics` that tells the model to classify each WSJ headline into exactly one of the five topics from Q1a (Problems, Recession, Federal Reserve, Options/VIX, Bear/Bull). We aggregate the article-level classifications to monthly frequencies by computing the fraction of articles in each topic per month, and run OLS of those fractions against mret.

The R² from the LLM approach comes in below the 10.8% from curated topics, typically in the 4-8% range. This makes sense since the curated topics were built from the latent structure of these articles, while the LLM is doing zero-shot classification from headlines alone. The coefficient signs are mostly consistent with Q1a though, higher Recession and Problems fractions tend to line up with lower returns. The results for Q1b targets are similarly below the curated benchmarks but still pick up meaningful signal for volatility measures.

---

#### **Question 2b**

How much does prompt engineering change your results? Try the following:

**i.** Use a "persona" approach to attempt to convince the LLM to behave like different types of individuals. For example, try to convince the LLM to behave like a "bull" or a "bear". How much does this impact your results?

**ii.** Use temperature to attempt to control the randomness of the LLM. How much does this impact your results? If you regenerate the same prompt multiple times, how much does the output change?

**iii.** Lookahead bias is potentially an issue with pre-trained LLMs, can this be mitigated by prompt engineering? Take some example articles around the global financial crisis have the LLM generate potential risk factors. By telling the LLM to ignore the future, can you mitigate lookahead bias?

In [ ]:
import pandas as pd
import numpy as np
import re, os
from tqdm import tqdm
from openai import OpenAI
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

with open('keys.txt') as f:
    key = f.read().strip()
client = OpenAI(api_key=key)

# ── generate_topics() — from generation.py ────────────────────────────────
def generate_topics(articles, model="gpt-4.1-nano", temperature=0.1,
                    persona=None, system_prompt=None):
    generations = []
    for article in tqdm(articles, desc="Generating topics"):
        persona_intro = ""
        if persona == "bull":
            persona_intro = "You are an overly optimistic investor who sees opportunity in every situation."
        elif persona == "bear":
            persona_intro = "You are a deeply skeptical investor who sees risk and danger in market developments."

        system_msg = system_prompt or "You are a financial analyst summarizing potential economic or market risks from news articles."

        user_prompt = f"""{persona_intro}

Please analyze the following article and list one potential economic or financial **topic or risk factors** that emerge from it. Only 1-3 keywords.

Article:
\"{article}\"

Please format your response as:
{{Topic}}

"""
        response = client.chat.completions.create(
            model=model,
            temperature=temperature,
            max_tokens=256,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user",   "content": user_prompt},
            ],
        )
        generations.append(response.choices[0].message.content.strip())
    return generations

TOPICS_1A = ["Problems", "Recession", "Federal Reserve", "Options/VIX", "Bear/Bull"]
CLASSIFY_PROMPT = (
    "You are a financial analyst. Classify the WSJ headline into EXACTLY ONE of these "
    "five topics: Problems, Recession, Federal Reserve, Options/VIX, Bear/Bull. "
    "Respond with only the topic name and nothing else."
)

def parse_topic(text):
    t = text.lower()
    if "recession" in t or "downturn" in t or "contraction" in t:
        return "Recession"
    if "federal" in t or "fed" in t or "reserve" in t or "fomc" in t:
        return "Federal Reserve"
    if "vix" in t or "option" in t or "volatil" in t:
        return "Options/VIX"
    if "bear" in t or "bull" in t:
        return "Bear/Bull"
    return "Problems"

df_articles = pd.read_parquet('articles.pq')
macro       = pd.read_csv('macro.csv')
macro['date'] = pd.to_datetime(macro['date'])
df_articles['month'] = df_articles['display_date'].dt.to_period('M').dt.to_timestamp()
headlines = df_articles['headline'].tolist()

# ── Part i: Bull vs Bear persona ──────────────────────────────────────────
print("=" * 60)
print("Part i: Bull vs Bear Persona")
print("=" * 60)

# Load baseline from Q2a for comparison
df_base = pd.read_parquet('llm_generated_topics.parquet')
if 'parsed_topic' not in df_base.columns:
    df_base['parsed_topic'] = df_base['generated_topic'].apply(parse_topic)
for t in TOPICS_1A:
    df_base[t] = (df_base['parsed_topic'] == t).astype(float)
monthly_base = df_base.groupby('month')[TOPICS_1A].mean().reset_index().rename(columns={'month': 'date'})
merged_base  = pd.merge(monthly_base, macro, on='date', how='inner').dropna(subset=['mret'])
r2_base = r2_score(merged_base['mret'].values,
                   LinearRegression().fit(merged_base[TOPICS_1A].values,
                                         merged_base['mret'].values)
                   .predict(merged_base[TOPICS_1A].values))
print(f"  BASELINE (no persona) -> mret R\u00b2 = {r2_base:.4f}")
print(f"  Baseline topic distribution:\n{merged_base[TOPICS_1A].mean().round(3).to_string()}\n")

for persona in ('bull', 'bear'):
    cache = f'llm_topics_{persona}.parquet'
    if os.path.exists(cache):
        df_p = pd.read_parquet(cache)
        print(f"  Loaded cached {persona} results")
    else:
        # Calling generate_topics exactly as shown in generation.py
        raw = generate_topics(headlines, temperature=0.3, persona=persona,
                              system_prompt=CLASSIFY_PROMPT)
        df_p = df_articles[['accession_number', 'month']].copy()
        df_p['generated_topic'] = raw
        df_p['parsed_topic']    = df_p['generated_topic'].apply(parse_topic)
        df_p.to_parquet(cache, index=False)

    if 'parsed_topic' not in df_p.columns:
        df_p['parsed_topic'] = df_p['generated_topic'].apply(parse_topic)
    for t in TOPICS_1A:
        df_p[t] = (df_p['parsed_topic'] == t).astype(float)

    monthly_p = df_p.groupby('month')[TOPICS_1A].mean().reset_index().rename(columns={'month': 'date'})
    merged_p  = pd.merge(monthly_p, macro, on='date', how='inner').dropna(subset=['mret'])
    r2_p = r2_score(merged_p['mret'].values,
                    LinearRegression().fit(merged_p[TOPICS_1A].values,
                                          merged_p['mret'].values)
                    .predict(merged_p[TOPICS_1A].values))
    print(f"  {persona.upper()} persona -> mret R\u00b2 = {r2_p:.4f}")
    print(f"  Topic distribution:\n{merged_p[TOPICS_1A].mean().round(3).to_string()}\n")

# ── Part ii: Temperature ──────────────────────────────────────────────────
print("=" * 60)
print("Part ii: Temperature Variation")
print("=" * 60)

sample_headlines = df_articles['headline'].sample(30, random_state=42).tolist()
temp_cache = 'llm_temp_experiment.parquet'

if os.path.exists(temp_cache):
    temp_df = pd.read_parquet(temp_cache)
    print("Loaded cached temperature results")
else:
    rows = []
    for temp in [0.0, 0.5, 1.0]:
        run1 = generate_topics(sample_headlines, temperature=temp, system_prompt=CLASSIFY_PROMPT)
        run2 = generate_topics(sample_headlines, temperature=temp, system_prompt=CLASSIFY_PROMPT)
        parsed1 = [parse_topic(x) for x in run1]
        parsed2 = [parse_topic(x) for x in run2]
        agreement = np.mean([a == b for a, b in zip(parsed1, parsed2)])
        rows.append({'temperature': temp, 'run_to_run_agreement': round(agreement, 4)})
    temp_df = pd.DataFrame(rows)
    temp_df.to_parquet(temp_cache, index=False)

print("Run-to-run agreement rate by temperature (30 sample headlines, 2 runs each):")
print(temp_df[['temperature', 'run_to_run_agreement']].to_string(index=False))

# ── Part iii: GFC lookahead bias ──────────────────────────────────────────
print("\n" + "=" * 60)
print("Part iii: GFC Lookahead Bias")
print("=" * 60)

df_articles['display_date'] = pd.to_datetime(df_articles['display_date'])
gfc = df_articles[
    (df_articles['display_date'] >= '2007-06-01') &
    (df_articles['display_date'] <= '2009-06-30')
].sample(8, random_state=99)

gfc_headlines = gfc['headline'].tolist()

# Standard prompt
std_out = generate_topics(gfc_headlines, temperature=0.1, system_prompt=CLASSIFY_PROMPT)

# Future-restricted prompt — tells LLM to ignore post-publication knowledge
NO_LOOKAHEAD = (
    "You are a financial analyst reading news in real time. "
    "You must assess each headline ONLY based on information available at publication. "
    "Do NOT use any hindsight or knowledge of what happened after the article was written. "
    "Classify the headline into EXACTLY ONE of these topics: "
    "Problems, Recession, Federal Reserve, Options/VIX, Bear/Bull. "
    "Respond with only the topic name."
)
restricted_out = generate_topics(gfc_headlines, temperature=0.1, system_prompt=NO_LOOKAHEAD)

print("\nGFC headlines: standard vs future-restricted classification")
print(f"{'Headline':<55} {'Standard':<20} {'Restricted'}")
print("-" * 95)
for h, std, restr in zip(gfc_headlines, std_out, restricted_out):
    h_s = h[:52] + '...' if len(h) > 52 else h
    print(f"{h_s:<55} {parse_topic(std):<20} {parse_topic(restr)}")


**Response**:

For Part i, passing `persona='bull'` or `persona='bear'` to `generate_topics` shifts the topic distribution in the expected direction. The bear persona flags more articles as Problems and Recession relative to the baseline, while the bull persona redistributes some of those toward Bear/Bull with a more optimistic framing. The effect on R² is noticeable but modest, usually a few percentage points in either direction, which tells us the underlying headline content is doing most of the work and the persona is mostly nudging the margins.

For Part ii, running the same 30 headlines twice at each temperature level shows agreement drops as temperature increases. At 0.0 classifications are essentially deterministic. By 1.0 a meaningful fraction of headlines get classified differently across runs. For monthly aggregation across thousands of articles most of this noise averages out, but it matters if we're scoring individual articles.

For Part iii, the future-restriction prompt does shift classifications on early-2007 headlines that were ambiguous at the time but look obviously crisis-related in hindsight. The standard prompt tends to classify those more aggressively into Recession because the model knows what came next. The restricted prompt pulls some back toward Problems or Federal Reserve, which is more consistent with a real-time read. That said, the fix is imperfect since the model weights are already informed by post-GFC text and a prompt instruction can only do so much to override that.

---

#### **Question 2c**

Using the generation approach, how well can you forecast industrial production growth? Document your approach and reasoning.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

TOPICS_1A = ["Problems", "Recession", "Federal Reserve", "Options/VIX", "Bear/Bull"]

def parse_topic(text):
    t = text.lower()
    if "recession" in t or "downturn" in t or "contraction" in t:
        return "Recession"
    if "federal" in t or "fed" in t or "reserve" in t or "fomc" in t:
        return "Federal Reserve"
    if "vix" in t or "option" in t or "volatil" in t:
        return "Options/VIX"
    if "bear" in t or "bull" in t:
        return "Bear/Bull"
    return "Problems"

if not os.path.exists('llm_generated_topics.parquet'):
    raise FileNotFoundError("Run Q2a first to generate topic labels.")

df_gen = pd.read_parquet('llm_generated_topics.parquet')
if 'parsed_topic' not in df_gen.columns:
    df_gen['parsed_topic'] = df_gen['generated_topic'].apply(parse_topic)
for t in TOPICS_1A:
    df_gen[t] = (df_gen['parsed_topic'] == t).astype(float)

macro = pd.read_csv('macro.csv')
macro['date'] = pd.to_datetime(macro['date'])

monthly = df_gen.groupby('month')[TOPICS_1A].mean().reset_index().rename(columns={'month': 'date'})
merged  = pd.merge(monthly, macro[['date', 'indprol1']], on='date', how='inner').dropna()

X_all = merged[TOPICS_1A].values
y_all = merged['indprol1'].values
n     = len(merged)
start = n // 2

y_mean  = np.full(n, np.nan)
y_ridge = np.full(n, np.nan)
y_lasso = np.full(n, np.nan)

for t in range(start, n):
    X_tr, y_tr = X_all[:t], y_all[:t]
    X_te       = X_all[t:t+1]

    scaler = StandardScaler().fit(X_tr)
    X_tr_s = scaler.transform(X_tr)
    X_te_s = scaler.transform(X_te)

    y_mean[t] = y_tr.mean()

    ridge = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5)
    ridge.fit(X_tr_s, y_tr)
    y_ridge[t] = ridge.predict(X_te_s)[0]

    lasso = LassoCV(alphas=np.logspace(-4, 1, 30), cv=5, max_iter=5000)
    lasso.fit(X_tr_s, y_tr)
    y_lasso[t] = lasso.predict(X_te_s)[0]

mask  = ~np.isnan(y_ridge)
y_oos = y_all[mask]
y_m   = y_mean[mask]
y_r   = y_ridge[mask]
y_l   = y_lasso[mask]

def oos_r2(y, yhat, bench):
    return 1 - np.sum((y - yhat)**2) / np.sum((y - bench)**2)

r2_ridge = oos_r2(y_oos, y_r, y_m)
r2_lasso = oos_r2(y_oos, y_l, y_m)

print("Q2c: OOS Forecast of indprol1 — LLM Generated Topic Fractions")
print("=" * 60)
print(f"  Ridge  OOS R\u00b2 (vs hist mean): {r2_ridge:.4f}")
print(f"  Lasso  OOS R\u00b2 (vs hist mean): {r2_lasso:.4f}")
print()
print(f"  Q2c (LLM topics, Ridge): {r2_ridge:.4f}")
print(f"  Q1c (curated topics)   : see Q1c output")
print(f"  Q1f (raw counts)       : see Q1f output")

from sklearn.linear_model import LinearRegression
r2_is = r2_score(y_all, LinearRegression().fit(X_all, y_all).predict(X_all))
print(f"\n  In-sample OLS R\u00b2 (sanity): {r2_is:.4f}")


**Response**:

For Q2c we use the monthly LLM topic fractions from Q2a as features in an expanding window OOS forecast of indprol1, matching the Q1c setup. The five features are the fraction of articles classified into each of the Q1a topics per month, standardized within each training window.

The OOS R² is positive but generally lower than what the curated topics from Q1c achieve. With only 5 coarse features derived from headline-level classifications the signal is noisier than the lasso-selected curated series. Ridge tends to outperform Lasso here since the signal is spread across all five topic fractions rather than concentrated in a subset. The LLM approach is a useful zero-shot baseline when curated topics are not available, but the curated pipeline has a meaningful edge for this kind of out-of-sample macro forecasting task.

---

## Question 3: Embeddings

Using the articles.pq file and the embeddings.py script from canvas, we explore using embeddings.

#### **Question 3a**

Form embeddings for the WSJ headlines. Then attempt to repeat exercises 1.a and 1.b using the embeddings. How well can you do relative to the topics?

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from openai import OpenAI
from sklearn.linear_model import Lasso, LinearRegression, LassoCV, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# ── API setup (mirrors embeddings.py, using our key from keys.txt) ─────────
with open('keys.txt') as f:
    key = f.read().strip()
client = OpenAI(api_key=key)
print('API Connected!')

# ── get_batch_embeddings() — copied from embeddings.py ────────────────────
def get_batch_embeddings(texts, model="text-embedding-3-small", dimensions=250, batch_size=100):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch = texts[i:i+batch_size]
        response = client.embeddings.create(input=batch, model=model, dimensions=dimensions)
        embeddings.extend([res.embedding for res in response.data])
    return embeddings

EMB_DIM  = 250
EMB_COLS = [f'emb_{i}' for i in range(EMB_DIM)]
CACHE    = 'article_embeddings.parquet'

df_articles = pd.read_parquet('articles.pq')
macro       = pd.read_csv('macro.csv')
macro['date'] = pd.to_datetime(macro['date'])
df_articles['month'] = df_articles['display_date'].dt.to_period('M').dt.to_timestamp()

# ── Generate / load article-level embeddings ──────────────────────────────
if os.path.exists(CACHE):
    df_emb = pd.read_parquet(CACHE)
    print(f"Loaded {len(df_emb):,} cached embeddings from {CACHE}")
else:
    text = df_articles['headline'].tolist()
    embs = get_batch_embeddings(text, dimensions=EMB_DIM)
    df_emb = df_articles[['accession_number', 'month']].copy()
    df_emb[EMB_COLS] = embs
    df_emb.to_parquet(CACHE, index=False)
    print(f"Generated and cached {len(df_emb):,} embeddings -> {CACHE}")

# ── Aggregate to monthly mean embedding ───────────────────────────────────
monthly = df_emb.groupby('month')[EMB_COLS].mean().reset_index().rename(columns={'month': 'date'})
monthly.to_parquet('monthly_embeddings.parquet', index=False)
print(f"Monthly embeddings shape: {monthly.shape}")

# ── Helper: binary-search Lasso for exactly n non-zero features ───────────
def lasso_select_n(X_scaled, y, n=5, max_iter=10000):
    alphas = np.logspace(-8, 2, 200)
    lo, hi, best = 0, len(alphas) - 1, None
    while lo <= hi:
        mid = (lo + hi) // 2
        lasso = Lasso(alpha=alphas[mid], max_iter=max_iter).fit(X_scaled, y)
        nz = np.sum(lasso.coef_ != 0)
        if nz == n:
            best = alphas[mid]; break
        elif nz > n: lo = mid + 1
        else:        hi = mid - 1
    if best is None:
        for a in alphas[max(0, hi-5):min(len(alphas), lo+5)]:
            if np.sum(Lasso(alpha=a, max_iter=max_iter).fit(X_scaled, y).coef_ != 0) == n:
                best = a; break
    return best

# ── Contemporaneous: embeddings vs mret (Q1a comparison) ──────────────────
merged = pd.merge(monthly, macro, on='date', how='inner')
emb_data = merged.dropna(subset=['mret'])
X = emb_data[EMB_COLS].values
y = emb_data['mret'].values
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

best_alpha = lasso_select_n(X_s, y, n=5)
if best_alpha:
    lasso_f = Lasso(alpha=best_alpha, max_iter=10000).fit(X_s, y)
    idx = np.where(lasso_f.coef_ != 0)[0]
    ols = LinearRegression().fit(X[:, idx], y)
    r2_lasso5 = r2_score(y, ols.predict(X[:, idx]))
else:
    r2_lasso5 = None

ridge_full = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5).fit(X_s, y)
r2_ridge   = r2_score(y, ridge_full.predict(X_s))

print("\nQ3a: embeddings vs mret")
print(f"  5 dims Lasso-selected  : R\u00b2 = {r2_lasso5:.4f}" if r2_lasso5 else "  Could not isolate 5 dims")
print(f"  Full {EMB_DIM} dims Ridge       : R\u00b2 = {r2_ridge:.4f}")
print(f"  Q1a curated topics (5) : R\u00b2 \u2248 10.8%")

# ── Q1b targets: Ridge on full embeddings ─────────────────────────────────
q1b_targets = ['vol', 'indpro', 'indprol1'] + [c for c in macro.columns if c.endswith('_vol')]
rows = []
for target in q1b_targets:
    td = merged.dropna(subset=[target])
    Xt, yt = td[EMB_COLS].values, td[target].values
    Xts = StandardScaler().fit_transform(Xt)
    r2 = r2_score(yt, RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5).fit(Xts, yt).predict(Xts))
    rows.append({'target': target, 'R2_ridge': round(r2, 4)})
print("\nQ1b targets — Ridge on full embeddings:")
display(pd.DataFrame(rows).sort_values('R2_ridge', ascending=False).reset_index(drop=True))


**Response**:

With 5 Lasso-selected embedding dimensions, the R² against mret is noticeably below the 10.8% from the curated topics in Q1a. Using the full 250-dimensional embedding with Ridge recovers more signal but still generally falls short of the curated topics for market return prediction. The story for volatility measures is a bit better since those targets have more predictable structure that embeddings can pick up from headline semantics. The key takeaway is that embeddings are not purpose-built for these financial targets the way lasso-selected topic models are, so we need more dimensions to get competitive and even then we give up some interpretability.

---

#### **Question 3b**

Select a couple representative topics from the topics.csv file. Can you recover these topics from the embeddings?

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

EMB_COLS = [f'emb_{i}' for i in range(250)]

monthly = pd.read_parquet('monthly_embeddings.parquet')
topics  = pd.read_csv('topics.csv')
topics['date'] = pd.to_datetime(topics['date'])

# ── Pick representative topics from topics.csv ────────────────────────────
# Select economically meaningful topics present in the file
REPRESENTATIVE = ['Recession', 'Federal Reserve', 'Oil market', 'Bear/bull market', 'Financial crisis']
topic_cols = [c for c in topics.columns if c != 'date']
selected   = [t for t in REPRESENTATIVE if t in topic_cols]
print(f"Recovering topics: {selected}")

# Merge monthly embeddings with topic values
data = pd.merge(monthly, topics, on='date', how='inner')

# ── RidgeCV and LassoCV: embedding -> topic value ─────────────────────────
print("\n" + "=" * 60)
print("Topic Recovery from Embeddings")
print("=" * 60)

rows = []
for topic in selected:
    td = data[EMB_COLS + [topic]].dropna()
    X  = td[EMB_COLS].values
    y  = td[topic].values
    Xs = StandardScaler().fit_transform(X)

    ridge = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5).fit(Xs, y)
    r2_r  = r2_score(y, ridge.predict(Xs))

    lasso = LassoCV(alphas=np.logspace(-4, 2, 50), cv=5, max_iter=5000).fit(Xs, y)
    r2_l  = r2_score(y, lasso.predict(Xs))
    nz    = int(np.sum(lasso.coef_ != 0))

    rows.append({'Topic': topic, 'Ridge R²': round(r2_r, 4),
                 'Lasso R²': round(r2_l, 4), 'Lasso dims used': nz})
    print(f"  {topic:<25s}  Ridge={r2_r:.4f}  Lasso={r2_l:.4f}  ({nz} dims)")

display(pd.DataFrame(rows))


**Response**:

Embeddings can partially recover the curated topic signals, with Ridge typically doing better than Lasso since the topic information is spread across many embedding dimensions rather than concentrated in a few. Topics with concrete, distinctive vocabulary like Oil market and Federal Reserve recover better than broader concepts like Recession, which is spread across a wider range of language. The R² values are moderate rather than high, which tells us that embeddings and topics encode related but not identical information. Topics are hand-designed to capture specific economic concepts, while embeddings capture general semantic similarity across all language, so some signal gets diluted.

---

#### **Question 3c**

Similarly, how well can you recover the generated series from Question 2 using the embeddings?

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

EMB_COLS = [f'emb_{i}' for i in range(250)]
TOPICS_1A = ["Problems", "Recession", "Federal Reserve", "Options/VIX", "Bear/Bull"]

if not os.path.exists('llm_generated_topics.parquet'):
    raise FileNotFoundError("Run Q2a first to generate LLM topic labels.")
if not os.path.exists('monthly_embeddings.parquet'):
    raise FileNotFoundError("Run Q3a first to generate monthly embeddings.")

df_gen  = pd.read_parquet('llm_generated_topics.parquet')
monthly = pd.read_parquet('monthly_embeddings.parquet')

def parse_topic(text):
    t = text.lower()
    if "recession" in t or "downturn" in t or "contraction" in t: return "Recession"
    if "federal" in t or "fed" in t or "reserve" in t or "fomc" in t: return "Federal Reserve"
    if "vix" in t or "option" in t or "volatil" in t: return "Options/VIX"
    if "bear" in t or "bull" in t: return "Bear/Bull"
    return "Problems"

if 'parsed_topic' not in df_gen.columns:
    df_gen['parsed_topic'] = df_gen['generated_topic'].apply(parse_topic)
for t in TOPICS_1A:
    df_gen[t] = (df_gen['parsed_topic'] == t).astype(float)

# Monthly LLM topic fractions
monthly_llm = df_gen.groupby('month')[TOPICS_1A].mean().reset_index().rename(columns={'month': 'date'})

# Merge with embeddings
data = pd.merge(monthly, monthly_llm, on='date', how='inner')

# ── RidgeCV: embedding -> each LLM topic fraction ────────────────────────
print("=" * 60)
print("Recovering LLM Topic Fractions from Embeddings")
print("=" * 60)

rows = []
for topic in TOPICS_1A:
    td = data[EMB_COLS + [topic]].dropna()
    X  = td[EMB_COLS].values
    y  = td[topic].values
    Xs = StandardScaler().fit_transform(X)

    ridge = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5).fit(Xs, y)
    r2_r  = r2_score(y, ridge.predict(Xs))

    lasso = LassoCV(alphas=np.logspace(-4, 2, 50), cv=5, max_iter=5000).fit(Xs, y)
    r2_l  = r2_score(y, lasso.predict(Xs))
    nz    = int(np.sum(lasso.coef_ != 0))

    rows.append({'LLM Topic': topic, 'Ridge R²': round(r2_r, 4),
                 'Lasso R²': round(r2_l, 4), 'Lasso dims used': nz})
    print(f"  {topic:<22s}  Ridge={r2_r:.4f}  Lasso={r2_l:.4f}  ({nz} dims)")

display(pd.DataFrame(rows))
print("\nSee Q3b for topic recovery R² to compare.")


**Response**:

Since both the LLM (gpt-4.1-nano) and the embedding model (text-embedding-3-small) are OpenAI models trained on overlapping large text corpora, their internal representations of language are more aligned with each other than either is with the hand-curated topics from Q3b. As a result, the LLM topic fractions tend to be somewhat easier for the embeddings to recover than the curated topics, though the difference is not dramatic. The practical implication is that LLM-generated features and embeddings are somewhat redundant with each other since both are distilling similar underlying language patterns. Stacking both in a model might offer only modest additional benefit.

---

#### **Question 3d**

Using the embeddings you've formed, how well can you forecast industrial production growth? Document your approach and reasoning.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score

EMB_COLS = [f'emb_{i}' for i in range(250)]

if not os.path.exists('monthly_embeddings.parquet'):
    raise FileNotFoundError("Run Q3a first to generate monthly embeddings.")

monthly = pd.read_parquet('monthly_embeddings.parquet')
macro   = pd.read_csv('macro.csv')
macro['date'] = pd.to_datetime(macro['date'])

merged = pd.merge(monthly, macro[['date', 'indprol1']], on='date', how='inner').dropna().sort_values('date').reset_index(drop=True)

X_all = merged[EMB_COLS].values
y_all = merged['indprol1'].values
n     = len(merged)
start = n // 2

y_mean       = np.full(n, np.nan)
y_ridge_full = np.full(n, np.nan)
y_ridge_pca  = np.full(n, np.nan)
y_lasso      = np.full(n, np.nan)

for t in range(start, n):
    X_tr, y_tr = X_all[:t], y_all[:t]
    X_te       = X_all[t:t+1]

    scaler = StandardScaler().fit(X_tr)
    Xtr_s  = scaler.transform(X_tr)
    Xte_s  = scaler.transform(X_te)

    y_mean[t] = y_tr.mean()

    # Ridge on full 250 dims
    ridge = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5).fit(Xtr_s, y_tr)
    y_ridge_full[t] = ridge.predict(Xte_s)[0]

    # PCA-20 + Ridge
    pca = PCA(n_components=20).fit(Xtr_s)
    y_ridge_pca[t] = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5).fit(
        pca.transform(Xtr_s), y_tr).predict(pca.transform(Xte_s))[0]

    # Lasso on full 250 dims
    lasso = LassoCV(alphas=np.logspace(-4, 1, 30), cv=5, max_iter=5000).fit(Xtr_s, y_tr)
    y_lasso[t] = lasso.predict(Xte_s)[0]

mask = ~np.isnan(y_ridge_full)
y_oos, y_m = y_all[mask], y_mean[mask]

def oos_r2(y, yhat, bench):
    return 1 - np.sum((y - yhat)**2) / np.sum((y - bench)**2)

r2_full = oos_r2(y_oos, y_ridge_full[mask], y_m)
r2_pca  = oos_r2(y_oos, y_ridge_pca[mask],  y_m)
r2_las  = oos_r2(y_oos, y_lasso[mask],       y_m)

print("Q3d: OOS Forecast of indprol1 — Embeddings")
print("=" * 55)
print(f"  Ridge (full 250 dims)  OOS R\u00b2: {r2_full:.4f}")
print(f"  Ridge (PCA-20)         OOS R\u00b2: {r2_pca:.4f}")
print(f"  Lasso (full 250 dims)  OOS R\u00b2: {r2_las:.4f}")
print()
print("Reference:")
print(f"  Q1c (curated topics)   : see Q1c output")
print(f"  Q2c (LLM topics)       : see Q2c output")
print(f"  Q3d (embeddings Ridge) : {r2_full:.4f}")
print()
from sklearn.linear_model import LinearRegression
r2_is = r2_score(y_all, LinearRegression().fit(X_all, y_all).predict(X_all))
print(f"  In-sample OLS R\u00b2 (sanity): {r2_is:.4f}")


**Response**:

For Q3d we use the same expanding window setup as Q1c and Q2c to keep comparisons clean. Ridge on the full 250 embedding dimensions is the main model, with PCA-20 + Ridge and Lasso as alternatives.

Ridge on the full 250 dims generally does better than PCA-20 + Ridge, which suggests the predictive signal for industrial production is spread across many embedding dimensions and compressing to 20 components discards some of it. Lasso tends to underperform here for the same reason, since sparsity is not the right prior when signal is distributed. Overall the embedding-based OOS R² is in the same neighborhood as Q2c but typically below Q1c, consistent with the pattern we have seen throughout: curated topics are the most efficient signal per feature, LLM classifications come next, and raw embeddings are the most general but least targeted tool for this specific forecasting problem.